# MA378 Lab 1 

Solving nonlinear equations with fited-point iteration.

## Instructions
1. Open this notebook on https://cloudjupyter.universityofgalway.ie/ or someother Jupyter/Python system you have access to.
1. Rename this file a suitable name, indicating your name. E.g., `MA385-Lab3-CFGauss.ipynb`.  **Do that now!**
2. In the `markdown` cell just below, give your name, ID number, email address, submission date, and a list of people you collaborated with. 
3. Add a note in the following cell confirming that you did not make use of Generative AI tools in completing this lab.
2. When you are finished, run the entire notebook from the top (`Kernel... Restart kernel and and run all cells`). Make sure there are no errors.
3. Go to and `File... Save and Export Notebook as` either a HTML (preferably) or PDF file. Any other format, including `.ipynb` scores.
5. Due: **5pm, Tuesday 13 October, 2026**


### Add your details here 
* Name:
* ID Number:
* `Email` address:
* Date:
* Who, if anybody, you worked with:

### Use of AI
<div class="alert alert-block alert-warning">
    <b><font size="+2">Statement on use, if any, of Generative AI tools</font></b> 
</div>

_Add a note here confirming that you did not make use of any Generative AI tools._


## How this lab works

### Collaboration
You are encouraged to work with class-mates on this lab. It is OK for people who've worked together to submit similar notebooks, providing that
1. You each upload your work
2. Your name and ID at the top of the first markdown cell. 
3. You list anyone you collaborated with.

### Collaboration
You are encouraged to work with class-mates on this lab. It is OK for people who've worked together to submit identical work, subject to the following conditions.
1. You submit a copy of the notebook
2. You give your name and ID at the top of the first makrdown cell. 
3. You list anyone you collaborated with.

### Grading
This part of the module is about encouraging engagement. It is not meant to be too demanding of your time, but to give you a framework to experiment (which is what we do in Computational Science).

* A notebook that shows a reasonable effort to answer all questions will get a grade of 3/3.
* If largely correct, but with some clear errors, or parts not attempted, will be 2/3.
* A notebook that shows little effort will score 1/3.
* Score 0/3 if you don't submit a HTML or PDF file.

### Using a different language
I've written this lab in Python, because most of the class are familiar with it. If you think you can complete it in some other language, then please do that. I'd be excited to see what you do. You still need to upload a `pdf` or `html` file. Contact Niall in advance if that is an issue.

Also, let know if you spot opertunities to make improvements.

## Some Python basics


In this lab we'll use the standard Python modules `numpy`, `scipy`, and `matplotlib`

Here is how to load them:

In [ ]:
import numpy as np
from scipy.optimize import root_scalar # nonlinear solver
import matplotlib.pyplot as plt

Here are other things you need to know for this lab, examples of which are provided:
1. How to define a function using `def`
2. How to plot using `plt.plot` 
3. How to use `f-strings`. I use them all the time. If want to brush up, see, e.g., https://www.datacamp.com/tutorial/python-f-string If you prefer to use other syntax that allows you to control how numbers are displayed (and, especially, the number of digits of precision used), then do that.

### Defining Functions

In this lab, we'll return to the old example of trying to estimate $\sqrt{2}$ by solving for the root of $$\underbrace{x^2-2}_{f(x)}=0$$
that is in $[1,2]$.

Let's define $f$:

In [ ]:
def f(x):
    return x**2 -2

### Using `scipy`

In this module, we are focused on understanding and analysing algorithms, and not so much in coding them (do `CS319` next semester if you want to learn that).

In any case, although we'll do a little implementation, we will happily use tools provided by `numpy` and `scipy` (and maybe other modules).

Now we solve for $f(\tau)=0$ using the `root_scalar` function, which is part of `scipy`. The arguments we'll give it are
* `f`, the function for which we want to solve $f(x)=0$.
* `bracket`, which is list of two floats, $a$ and $b$: the algorithm will search for the zero between $a$ and $b$.
* `xtol` what is the (absolute) tolerence for desired error. 

It can take other arguments, such as `Method`, which can be `bisect`, `Newton`, `secant`, and various others. 

In [ ]:
rs = root_scalar(f, bracket=[1, 2], xtol=1e-15) # solve using the `root_scalar` function
tau = rs.root
print(f"Estimate for tau is {tau:.15f}")

Check the error:

In [ ]:
print(f"Error: |tau - sqrt(2)|={np.abs(tau-np.sqrt(2)):.4e}")

### Plotting

Here is how to plot $f$, and $\tau$:

In [ ]:
x = np.linspace(0, 3, 401)  # 401 points between 0 and 3
plt.plot(x, f(x), tau, f(tau), 'r*')
plt.legend(['f(x)', '$\\tau$'])
plt.axhline(0, color='k', linewidth=1.5)  # x-axis
plt.axvline(0, color='k', linewidth=1.5)  # y-axis
plt.xlabel("x"), plt.ylabel("f(x)")
plt.grid(True)

## Fixed Point Iteration: 1st order

In this lab, we'll explore solving $f(x)=0$ using Fixed Point Iteration. We are interested in investigating the following

1. When we choose $g(x)$, at the very least we should verify that $f(\tau)=0 \leftrightarrow g(\tau)=\tau$.
2. Check if $g$ maps $[1,2]$ to $[1,2]$.
3. If possible, check if $g$ is a contraction
4. Verify that the method works
5. Try to establish the order of convergence.

### Try $g_1(x)= -\frac{x^2}{2}+x+1$

First, we'll define the function

In [ ]:
def g1(x):
    return -(x**2)/2 + x +1

#### Check its value at the value of $\tau$ computed above:

In [ ]:
print(f"tau={tau:.10f}, g1(tau)={g1(tau):.10f}")

#### $g_1: [1,2] \to [1,2]$?
For the iteration $x_{k+1}=g_1(x_k)$ to work for our problem, 
at the very least, we need $1 \leq g_1(x) \leq 2$ for all $x\in [1,2]$.

Let's check:

In [ ]:
x = np.linspace(1, 2, 201)  # 401 points between -1 and 3
plt.plot(x, x, '--', x, g1(x))
plt.legend(['x', 'g(x)'])
plt.xlabel('x'); plt.ylabel('y');
plt.grid(True)

#### Is $g_1$ a contraction?

We'd like to establish that $g_1$ is a contraction. That is, we'd like to demonstrate that there is some $L \in (0,1)$ such that $|g_1(\alpha) - g_1(\beta)| \leq L|\alpha - \beta|$ for all $\alpha, \beta \in [1,2]$. However, it is enough to show that $|g_1(x)| < 1$ for all $x$.

Since 
$g_1(x)= -\frac{x^2}{2}+x+1$, we can see that
$g'_1(x)= -x+1$.
So, if we ignore the minor detail that this gives $L \in [0,1]$, is fine. 

### EXERCISE 1: plot $g'$

Plot $|g'_1(x)|$ to show that $|g'_1(x)| \leq 1$ for $1 \leq x \leq 2$. 
You can use the explicit formula for $g'_1$ if you wish. But you might also like to investigate using `scipy.misc.derivative` (depreciated) or `np.gradient` to estimate the derivative for you:


In [ ]:
### YOUR CODE GOES HERE


#### Doing FPI

Finally, lets solve the problem using FPI. We should really write a function to implement this, but we'll be a little lazy for now, and just do 10 iterations:


In [ ]:
x1k = [] # will be a list of estimates
E1k = [] # will be list of errors
x1k.append(1.0)
E1k.append(np.abs(tau-x1k[-1]))
for k in range(10):   # choice of 10 is fairly arbitrary
    x1k.append(g1(x1k[-1]))
    E1k.append(np.abs(tau-x1k[-1]))

Best to observe convergence visually; we'll plot $\mathcal{E}_k$ against $k$ (the number of iterations) using a logarithmic scale on the $y$-axis.

In [ ]:
plt.semilogy(E1k, '--x')
plt.xlabel('k'); plt.ylabel('$\\mathcal{E}_k$');

So, the error decreases as $k$ in increases. It looks like the method is converges, albeit slowly. 

***

## Fixed Point Iteration: 2nd order

Let's try another approach, with $g_2(x) := \frac{x}{2}  + \frac{1}{x}.$

This is exactly the formula you would arrive at if you employed Newton's method for this problem: that is a fact that you should verify for yourself.


### Define $g_2$

In [ ]:
def g2(x):
    return(x/2.0 + 1.0/x)

### *EXERCISE 2:* Check that $g_2(\tau) = \tau$

In [ ]:
### Your code goes here


### EXERCISE 3: Check that $1< g_2(x)< 2$ for all $x\in [1,2]$

Now plot $g_2(x)$ to verify that $g_2$ maps $[1,2]$ to $[1,2]$

In [ ]:
## YOUR CODE GOES HERE


#### Convergence

We'll skip checking that $g_2$ is a contraction (do check that in your own time). 

Let's try a few iterations to check for convergence:

In [ ]:
x2k = [] # will be a list of estimates
E2k = [] # will be list of errors
x2k.append(1.0)
E2k.append(np.abs(tau-x2k[-1]))
for k in range(5):   
    x2k.append(g2(x2k[-1]))
    E2k.append(np.abs(tau-x2k[-1]))

Now let's plot the results:

In [ ]:
plt.semilogy(E1k, '--x', E2k, '--o')
plt.xlabel('k'); plt.ylabel('$\\mathcal{E}_k$');

If all has gone to plan, we should observe that, for this choice of $g_2$, the method converges more rapidly.

## A 3rd order method

For reasons we'll explore in the next Tutorial, we can choose of $g_3$ than converges  even more rapidly to $\tau$.


1. Define $g_3(x)= \frac{x^3+6x}{3x^2+2}$
2. Verify that it has $\tau = \sqrt{2}$ as a fixed point.
3. Verify that it maps $[1,2]$ to $[1,2]$
4. Compare the convergence of this method to those we get with the choices of $g_1$ and $g_2$ above.

### EXERCISE 4: define $g_3(x)$ and check that $g_3(\tau)=\tau$

In [ ]:
#### YOU CODE GOES HERE


In [ ]:
### Your code goes here


### EXERCISE 5: Check and plot convergence

In [ ]:
### Your code goes here


In [ ]:
### Your code goes here